# 💬 Day 04a: HLD — WhatsApp / Chat System

---

## 🎯 What You'll Master Today
- WebSocket real-time messaging
- Message delivery states (sent/delivered/read)
- Group messaging architecture
- Offline message storage
- End-to-end encryption overview

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  WHATSAPP / CHAT SYSTEM — ARCHITECTURE                          ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Core: Real-time 1:1 and group messaging                        ║
║                                                                   ║
║  1:1 Message Flow:                                               ║
║  ┌──────┐  WebSocket  ┌──────────┐  WebSocket  ┌──────┐       ║
║  │UserA │◄═══════════►│ Chat     │◄═══════════►│UserB │       ║
║  └──────┘             │ Server   │             └──────┘       ║
║                        └────┬─────┘                              ║
║                             │                                     ║
║                        ┌────▼─────┐                              ║
║                        │ Message  │                              ║
║                        │ Queue    │  (Kafka) for async delivery  ║
║                        └────┬─────┘                              ║
║                        ┌────▼─────┐                              ║
║                        │ Message  │  DB (Cassandra — write-heavy)║
║                        │ Store    │                              ║
║                        └──────────┘                              ║
║                                                                   ║
║  Key Insight: WebSocket = persistent bidirectional connection    ║
║  • HTTP: client→server only (poll for new messages = wasteful)  ║
║  • WebSocket: both directions, real-time push                   ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# 1. Scale Estimation
# ============================================================

print("""
📊 WHATSAPP SCALE ESTIMATION

Assumptions:
  • DAU: 2 billion (WhatsApp actual)
  • 40 messages/user/day average
  • Message size: 100 bytes (text), 1MB (media)
  • 5% messages have media

Message QPS:
  2B × 40 / 86400 ≈ 925,000 msgs/sec

Concurrent WebSocket connections:
  ~500M concurrent (at peak)
  Each connection uses ~10KB memory
  500M × 10KB = 5 TB → need ~5000 servers (1M conn each)

Storage (1 year — text only):
  2B × 40 × 100B × 365 ≈ 2.9 PB/year
  + media (5%): 2B × 40 × 0.05 × 1MB × 365 ≈ huge → media has TTL!

💡 WhatsApp deletes media from servers after delivery
   (stored on device only — E2E encrypted)
""")

In [ ]:
# ============================================================
# 2. Message Delivery & Status
# ============================================================

print("""
✉️ MESSAGE DELIVERY STATES

  ✓  SENT      → message reached server
  ✓✓ DELIVERED  → message reached recipient's device
  ✓✓ READ (blue)→ recipient opened the chat

Flow:
  1. UserA sends message → Chat Server
  2. Server stores in DB → sends ACK (✓ sent) to UserA
  3. If UserB online:
     Server pushes via WebSocket → UserB
     UserB device sends ACK → Server
     Server sends delivery receipt (✓✓) to UserA
  4. If UserB opens chat:
     UserB sends "read" receipt → Server → UserA (blue ✓✓)
  5. If UserB OFFLINE:
     Message stored in "offline queue" in DB
     When UserB reconnects → pull all pending messages
     Then send delivery receipts

  ┌────────────────────────────────────────────────────────┐
  │ Scenario        │ Protocol        │ Delivery          │
  ├────────────────────────────────────────────────────────┤
  │ Both online      │ WebSocket push  │ Instant           │
  │ Receiver offline │ Store → pull    │ When reconnects   │
  │ Sender offline   │ Queue on device │ When reconnects   │
  │ Both offline     │ Store on device │ Sequential sync   │
  └────────────────────────────────────────────────────────┘
""")

In [ ]:
# ============================================================
# 3. Group Messaging
# ============================================================

print("""
👥 GROUP MESSAGING

Challenge: 1 message → N members need to receive it

Approach 1: Fan-out per message
  • Sender sends 1 message → Server copies to each member
  • Group of 256: 1 send = 255 deliveries
  • Pro: Simple
  • Con: 255× write amplification

Approach 2: Store once, deliver many (WhatsApp's approach)
  • Store message ONCE with group_id
  • Track per-member delivery/read status separately
  • Members pull messages they haven't seen
  
  Messages Table:
  ┌────────────────────────────────────────────┐
  │ msg_id │ group_id │ sender │ text │ ts     │
  │        │ INDEX    │        │      │ INDEX  │
  └────────────────────────────────────────────┘
  
  Delivery Status Table:
  ┌──────────────────────────────────────────┐
  │ msg_id │ user_id │ delivered │ read_at   │
  │        │         │ BOOL      │ TIMESTAMP │
  └──────────────────────────────────────────┘

Group Max: WhatsApp = 1024, Telegram = 200K
  • Small groups (<100): push to all online members
  • Large groups (>1000): use pub/sub (Kafka topic per group)

Last Seen / Typing:
  • Don't persist! Just relay via WebSocket
  • Typing indicator: send event, auto-expire in 5 sec
""")

In [ ]:
# ============================================================
# 4. Connection Management & Architecture
# ============================================================

print("""
╔════════════════════════════════════════════════════════════════╗
║  CHAT SYSTEM FULL ARCHITECTURE                               ║
╠════════════════════════════════════════════════════════════════╣
║                                                                ║
║  ┌─────────┐  WS  ┌──────────┐                              ║
║  │ Client  │◄════►│ WS       │    Connection Manager:       ║
║  │ (app)   │      │ Gateway  │    • user_id → server_id     ║
║  └─────────┘      │ (Nginx)  │    • Stored in Redis/Zookeeper║
║                    └────┬─────┘    • When user connects:     ║
║                         │           register(user, server)    ║
║                    ┌────▼─────┐                              ║
║                    │  Chat    │    Routing:                   ║
║                    │  Server  │    A→B: lookup B's server    ║
║                    │  Cluster │    → forward via internal RPC ║
║                    └────┬─────┘                              ║
║              ┌──────────┼───────────┐                        ║
║         ┌────▼───┐ ┌───▼────┐ ┌────▼───┐                   ║
║         │ Redis  │ │ Kafka  │ │Cassandra│                   ║
║         │Session │ │Message │ │Messages │                   ║
║         │+ Conn  │ │Queue   │ │+ Groups │                   ║
║         └────────┘ └────────┘ └─────────┘                   ║
║                                                                ║
║  Why Cassandra for messages?                                  ║
║  • Write-heavy workload (1M writes/sec)                      ║
║  • Partition key = (chat_id) for co-located messages         ║
║  • Clustering key = (timestamp) for ordered retrieval        ║
║  • Easy horizontal scaling                                   ║
║                                                                ║
║  End-to-End Encryption (E2E):                                ║
║  • Signal Protocol (used by WhatsApp)                        ║
║  • Each device has public/private key pair                   ║
║  • Server CANNOT read messages (only relay encrypted blobs)  ║
║  • Key exchange: Diffie-Hellman                              ║
║  • Server stores encrypted messages only                     ║
║                                                                ║
╚════════════════════════════════════════════════════════════════╝
""")

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Design WhatsApp | WebSocket, Cassandra for msgs, Redis for connection map, Kafka for async |
| 2 | WebSocket vs long polling? | WS=bidirectional, persistent. LP=request-response, reconnect overhead |
| 3 | Message delivery states? | Sent(server ACK), Delivered(device ACK), Read(user opened chat) |
| 4 | Offline messages? | Store in DB/queue. Pull on reconnect. Push ACKs back |
| 5 | Group messaging at scale? | Store once, deliver many. Large groups use pub/sub (Kafka topics) |
| 6 | How E2E encryption works? | Signal protocol. Server relays encrypted blobs. Can't read content |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • WebSocket = persistent bidirectional connection        │
## │  • Connection manager: user → server mapping (Redis)     │
## │  • Three delivery states: sent/delivered/read             │
## │  • Offline: store in queue, deliver on reconnect         │
## │  • Cassandra: write-heavy, partition by chat_id          │
## │  • E2E: server is blind relay, Signal protocol           │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **HLD — Netflix / YouTube (Video Streaming)**